In [7]:
import sys
sys.path.append(r'd:\VSCode\neylon-ai\primary-server')

In [8]:
import os
print(os.getcwd())

d:\VSCode\neylon-ai\primary-server\resume_assistant


In [ ]:
pip install PyMuPDF

In [10]:
import fitz

path = 'lib/data/Hruthik_m.pdf'
print(os.path.exists(path))

True


In [12]:
doc = fitz.open(path)
text = ""
for page in doc:
    text += page.get_text("text")
    # for b in blocks:
    #     text += b[4].strip() + "\n\n"

with open("lib/data/output.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("✅ Text with layout preserved saved to output.txt")

✅ Text with layout preserved saved to output.txt


In [ ]:
doc = fitz.open(path)
text = ""
for page in doc:
    text += page.get_text("text")

print(text)
with open("lib/data/output.txt", "w", encoding="utf-8") as f:
    f.write(text)

print("✅ Text with layout preserved saved to output.txt")

In [21]:
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
load_dotenv()

GEMINI_API_KEY=os.getenv("GOOGLE_API_KEY")

base_model = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    temperature=0.4,
    max_retries=2,
    google_api_key=GEMINI_API_KEY
)

In [22]:
SYSTEM_PROMPT="""
You are an expert Resume Role Adaptation Assistant. Your goal is to intelligently tailor an existing resume to match a given job role or job description without losing structure, section order, or factual accuracy.

Rules:
0. Preserve Original Structure:
   - Keep the resume’s sections in the same order as the original.
   - Keep all section titles exactly as they appear
1. Selective Modification Based on Role:
   - Adjust the section lines under each experience/project to align with the provided job role.
   - Only make modifications necessary to improve relevance (e.g., emphasizing specific technologies, soft skills, or responsibilities that match the job role).
   - Do not invent or remove any real experience, project, or education unless explicitly instructed by the user.
2. Preserve Authenticity:
   - Keep all factual details (names, dates, titles, metrics, and projects) intact unless the user provides new information.
   - Do not alter quantitative achievements or company names.
3. Integrate Additional Content if Provided:
   - If the user provides extra material, seamlessly insert it into the most relevant section — without breaking the structure.
4. Tone and Style:
   - Maintain a professional, concise, and impact-driven tone.
   - Use action verbs and role-aligned keywords relevant to the target job description.
   - Focus on clarity and alignment with the target role’s requirements.
5. Output Format:
   - Return the entire modified resume text.
   - Ensure section headings, bullet structure, and layout are clearly preserved.
   - No explanations, comments, or notes — only the final formatted resume.

Input Format Expected:
- Resume Extracted Text: (full existing resume in plain text)
- Target Role or Job Description: (details of the new role)
- (Optional) Additional Info / Projects: any new content to integrate.

Generate a fully rewritten resume optimized for the target job role while preserving all original sections, order, and authenticity.
"""

In [23]:
import tiktoken 

encoding_model = "cl100k_base"
def get_encoding(text: str)->int:
    encoding = tiktoken.get_encoding(encoding_model)
    return len(encoding.encode(text))

In [24]:
print(get_encoding(SYSTEM_PROMPT))

390


In [25]:
user_prompt="""
Job Title: Full Stack Developer
Company: Cloudify Tech
Location: Remote

Description:
We are seeking a Full Stack Developer proficient in JavaScript and modern web frameworks. The ideal candidate will have hands-on experience developing scalable backend APIs, secure authentication systems, and interactive UIs.

Additional project:

CloudMetrics Dashboard
- Designed and developed a cloud monitoring dashboard using Next.js and Express.
- Integrated AWS CloudWatch APIs for real-time data visualization.
- Deployed application using Docker on AWS EC2.
"""

In [26]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": f"old_resume: {text}\nuser: {user_prompt}"}
]

In [27]:
response = base_model.invoke(messages)

In [28]:
with open("lib/data/response.txt", "w", encoding="utf-8") as f:
    f.write(response.content)

In [ ]:
print(response.content)

In [46]:
RESUME_EXTRACTOR_PROMPT = """
You are a highly accurate resume data extraction model.  
Your task is to read a raw resume text and extract all relevant information into a structured JSON object strictly following the schema below.

Rules:
- Always include all keys exactly as shown in the schema.
- If any field is missing or not found, set its value to the string `"null"`.
- Return Only Valid JSON — no explanations, no comments, no extra text.
- Preserve lists, nested structures, and null placeholders properly.
- Extract multiple entries where applicable into lists as per below schema (e.g., multiple experiences, projects, education items).

Output format:
{{
    "name": "<person_name>",
    "contact": {{
        "phone": "<contact_number>",
        "email": "<email_address>",
        "links": {{
            "LinkedIn": "<linkedin_url>",
            "GitHub": "<github_url>",
            # optional: add more if needed
            "<key>": "<value>"
        }}
    }},
    "skills": {{
        "title": "<section_name>",
        # Example: "Skills"
        "entries": {{
            "<skill_category>": "<skills_list_comma_separated>",
            # Example: "Languages": "Python, JavaScript, C++"
        }}
    }},
    "experience": {{
        "title": "<section_name>",
        "entries": [
            {{
                "role": "<job_role>",
                "company": "<job_company>",
                "duration": "<job_duration>",
                # Example: "April 2020 – June 2022", "April 2020 – Present"
                "location": "<job_location>",
                "highlights": [
                    "<job_highlight_point>",
                    "..."
                ]
            }}
        ]
    }},
    "projects": {{
        "title": "<section_name>",
        "entries": [
            {{
                "name": "<project_name>",
                "tech_stack": "<project_tech_stack>",
                # Example: "Python, React.js, VPS"
                "links": {{
                    "Live": "<project_live_url>",
                    "GitHub": "<project_github_url>",
                    # optional
                    "<key>": "<value>"
                }},
                "highlights": [
                    "<project_highlight_point>",
                    "..."
                ]
            }}
        ]
    }},
    "education": {{
        "title": "<section_name>",
        "entries": [
            {{
                "institution": "<institution_name>",
                "degree": "<degree_name>",
                # Example: "Bachelor of Technology in Mechanical Engineering"
                "duration": "<education_duration>",
                # Example: "Aug 2017 – Aug 2020"
                "location": "<education_location>"
            }}
        ]
    }}
}}
"""

In [48]:
print(get_encoding(RESUME_EXTRACTOR_PROMPT))

554


In [32]:
import os
from langchain_openai import ChatOpenAI

OPENAI_API_KEY=os.getenv("OPENAI_API_KEY")
openai_model = ChatOpenAI(model="gpt-4o-mini", temperature=0.4, api_key=OPENAI_API_KEY)

In [ ]:
response = openai_model.invoke([
    {"role": "system", "content": RESUME_EXTRACTOR_PROMPT},
    {"role": "user", "content": response.content}
])

In [ ]:
print(response.content)

In [40]:
import re
import json

def parse_json(raw_response):
    if not raw_response:
        return None
    match = re.search(r'\{.*\}', raw_response, re.S)
    if match:
        return json.loads(match.group(0))
    return None

In [43]:
resume_data = parse_json(response.content)

In [51]:
print(resume_data)
print(get_encoding(response.content))

{'name': 'Hruthik M', 'contact': {'phone': '+91 7483229386', 'email': 'mhrithik450@gmail.com', 'links': {'LinkedIn': 'null', 'GitHub': 'null'}}, 'skills': {'title': 'Technical Skills', 'entries': {'Frontend': 'React.js, Next.js, Tailwind CSS', 'AI Frameworks': 'Langchain, Langgraph, Chroma DB, Open AI, Google Gemini', 'Backend': 'Django, FastAPI, Express.js, Node.js, Postman API, Websockets, REST API’s', 'Databases': 'MongoDB, PostgreSQL', 'Cloud': 'Google Cloud Platform, Firebase, AWS', 'Languages': 'Python, JavaScript, C++', 'Tools': 'Git, GitHub, Vercel, Render, VPS, Figma, Cloud Run, Docker'}}, 'experience': {'title': 'Experience', 'entries': [{'role': 'Software Development Engineer', 'company': 'Codedale', 'duration': 'April 2025 – Present', 'location': 'India', 'highlights': ['Built and maintained 100+ robust backend endpoints using TypeScript and Next.js, reducing API response time by 90% and improving system reliability for 100k users.', 'Designed and implemented PostgreSQL sch

In [45]:
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.pdfgen import canvas
from reportlab.lib.units import inch
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import Paragraph, SimpleDocTemplate, Spacer, Table, TableStyle, HRFlowable
from reportlab.lib import colors

pdf_file = "lib/data/Hruthik_M_Genrated_Resume.pdf"
doc = SimpleDocTemplate(pdf_file, pagesize=A4, rightMargin=10, leftMargin=10, topMargin=2, bottomMargin=2)

styles = getSampleStyleSheet()
normal = styles["Normal"]

styles.add(ParagraphStyle(name="JobTitle", fontSize=11, leading=14, spaceBefore=6, spaceAfter=2, leftIndent=10))

story = []

# Name
story.append(Paragraph(resume_data["name"], ParagraphStyle(name="Name", fontSize=28, alignment=1, spaceAfter=8, leading=28, fontName="Times-Roman")))

# Contact
contact = resume_data.get("contact", {})
links = contact.get("links", {})

# Extract safely with fallbacks
phone = contact.get("phone", "")
email = contact.get("email", "")
linkedin = links.get("LinkedIn", "")
github = links.get("GitHub", "")

contact_parts = []
contact_parts.append(f"{phone}") if phone != "null" else ""
contact_parts.append(f"<link href='mailto:{email}' color='#085A8C' underline='true'>{email}</link>") if email != "null" else ""
contact_parts.append(f"<link href='{linkedin}' color='#085A8C' underline='true'>{linkedin}</link>") if linkedin != "null" else ""
contact_parts.append(f"<link href='{github}' color='#085A8C' underline='true'>{github}</link>") if github != "null" else ""

if len(contact_parts) > 3:
    contact_info = ("&nbsp;&nbsp;".join(contact_parts[:3]) + "<br/>" + "&nbsp;&nbsp;".join(contact_parts[3:]))
else:
    contact_info = "&nbsp;&nbsp;".join(contact_parts)
story.append(Paragraph(contact_info, ParagraphStyle(name="Contact", fontSize=12, alignment=1, spaceAfter=12, leading=17, fontName="Times-Roman")))

# Skills Section
story.append(Paragraph(f'<b>{resume_data["skills"]["title"]}</b>', ParagraphStyle(name="SectionTitle", fontSize=13, leading=16, spaceBefore=6, underlineWidth=1, fontName="Times-Roman")))
story.append(HRFlowable(width="100%", thickness=0.5, lineCap='round', color="#000000", spaceBefore=2, spaceAfter=2))

for key, value in resume_data["skills"]["entries"].items():
    story.append(Paragraph(f"<b>{key}:</b> {value}", ParagraphStyle(name="SkillPoints", fontSize=12, spaceBefore=2, spaceAfter=2, bulletIndent=10, leading=15, fontName="Times-Roman")))

# Experience Section
story.append(Paragraph(f'<b>{resume_data["experience"]["title"]}</b>', ParagraphStyle(name="SectionTitle", fontSize=13, leading=16, spaceBefore=6, underlineWidth=1, fontName="Times-Roman")))
story.append(HRFlowable(width="100%", thickness=0.5, lineCap='round', color="#000000", spaceBefore=2, spaceAfter=2))

for entry in resume_data.get("experience", {}).get("entries", []):
    role = entry.get("role", "")
    company = entry.get("company", "")
    duration = entry.get("duration", "")
    location = entry.get("location", "")
    highlights = entry.get("highlights", [])

    job_role = Paragraph(f'<b>{role}</b>', ParagraphStyle(name="JobRole", parent=normal, fontName="Times-Roman", fontSize=12))
    job_company = Paragraph(company, ParagraphStyle(name="JobCompany", parent=normal, fontName="Times-Roman", fontSize=12))
    job_duration = Paragraph(f'<b>{duration}</b>', ParagraphStyle(name="JobDuration", parent=normal, fontName="Times-Roman", fontSize=12, alignment=2))
    job_location = Paragraph(location, ParagraphStyle(name="JobLocation", parent=normal, fontName="Times-Roman", fontSize=12, alignment=2))

    data = [[job_role, job_duration], [job_company, job_location]]

    table = Table(data, colWidths=["70%", "30%"])

    table.setStyle(TableStyle([
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("ALIGN", (1, 0), (1, 0), "RIGHT"),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 0),
        ("LEFTPADDING", (0, 0), (-1, -1), 0),
        ("RIGHTPADDING", (0, 0), (-1, -1), 0),
        ("ROWSPACING", (0, 0), (-1, -1), 6)
    ]))
    story.append(table)

    if highlights:
        for point in highlights:
            story.append(Paragraph(f"<bullet>&bull;</bullet> {point}", ParagraphStyle(name="BulletPoints", fontSize=12, leftIndent=20, spaceBefore=2, spaceAfter=2, bulletIndent=10, leading=15, fontName="Times-Roman")))
    story.append(Spacer(1, 6))

# Projects Section
story.append(Paragraph(f'<b>{resume_data["projects"]["title"]}</b>', ParagraphStyle(name="SectionTitle", fontSize=13, leading=16, underlineWidth=1, fontName="Times-Roman")))
story.append(HRFlowable(width="100%", thickness=0.5, lineCap='round', color="#000000", spaceBefore=2, spaceAfter=2))

for entry in resume_data["projects"]["entries"]:
    project_name = Paragraph(f'<b>{entry["name"]}</b> | {entry["tech_stack"]}', ParagraphStyle(name="NameTechStack", parent=normal, fontName="Times-Roman", fontSize=12))
    
    live_link = entry.get("links", {}).get("Live")
    github_link = entry.get("links", {}).get("GitHub")

    links_html = ""
    if live_link:
        links_html += f"<link href='{live_link}' color='#085A8C' underline='true'>View Live</link>"
    if github_link:
        if links_html:
            links_html += " | "
        links_html += f"<link href='{github_link}' color='#085A8C' underline='true'>GitHub</link>"
    project_cta = Paragraph(links_html, ParagraphStyle(name="links_html", parent=normal, fontName="Times-Roman", fontSize=12, alignment=2))

    data = [[project_name, project_cta]]
    table = Table(data, colWidths=["75%", "25%"])
    table.setStyle(TableStyle([
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("ALIGN", (1, 0), (1, 0), "RIGHT"),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 0),
        ("LEFTPADDING", (0, 0), (-1, -1), 0),
        ("RIGHTPADDING", (0, 0), (-1, -1), 0),
        ("ROWSPACING", (0, 0), (-1, -1), 6)
    ]))
    story.append(table)

    for point in entry["highlights"]:
        story.append(Paragraph(f"<bullet>&bull;</bullet> {point}", ParagraphStyle(name="BulletPoints", fontSize=12, leftIndent=20, spaceBefore=2, spaceAfter=2, bulletIndent=10, leading=15, fontName="Times-Roman")))
    story.append(Spacer(1, 6))

# Education Section
story.append(Paragraph(f'<b>{resume_data["education"]["title"]}</b>', ParagraphStyle(name="education", fontSize=13, leading=16, spaceBefore=6, underlineWidth=1, fontName="Times-Roman")))
story.append(HRFlowable(width="100%", thickness=0.5, lineCap='round', color="#000000", spaceBefore=2, spaceAfter=2))

for entry in resume_data["education"]["entries"]:
    institution = entry.get("institution", "")
    duration = entry.get("duration", "")
    degree = entry.get("degree", "")
    location = entry.get("location", "")

    row1 = []
    row2 = []

    row1.append(Paragraph(f'<b>{entry["institution"]}</b>', ParagraphStyle(name="institution", parent=normal, fontName="Times-Roman", fontSize=12))) if institution != "null" else row1.append("")
    row1.append(Paragraph(f'<b>{entry["duration"]}</b>', ParagraphStyle(name="duration", parent=normal, fontName="Times-Roman", fontSize=12, alignment=2))) if duration != "null" else row1.append("")
    row2.append(Paragraph(f'{entry["degree"]}', ParagraphStyle(name="degree", parent=normal, fontName="Times-Roman", fontSize=12))) if degree != "null" else row2.append("")
    row2.append(Paragraph(f'{entry["location"]}', ParagraphStyle(name="location", parent=normal, fontName="Times-Roman", fontSize=12, alignment=2))) if location != "null" else row2.append("")

    data = [row1, row2]
    table = Table(data, colWidths=["75%", "25%"])
    table.setStyle(TableStyle([
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("ALIGN", (1, 0), (1, 0), "RIGHT"),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
        ("TOPPADDING", (0, 0), (-1, -1), 0),
        ("LEFTPADDING", (0, 0), (-1, -1), 0),
        ("RIGHTPADDING", (0, 0), (-1, -1), 0)
    ]))
    story.append(table)

doc.build(story)
print(f"✅ Resume saved as {pdf_file}")

✅ Resume saved as lib/data/Hruthik_M_Genrated_Resume.pdf


In [ ]:
# Example
resume_data = {
"name": "Hruthik M",
    "contact": {
        "phone": "+91 7483229386",
        "email": "mhrithik450@gmail.com",
        "links": {
            "LinkedIn": "linkedin.com/in/hruthik-m-3595a0329",
            "GitHub": "github.com/Hrithik450"
        }
    },
    "skills": {
        "title": "Skills",
        "entries": {
            "Languages": "JavaScript, C++, Java, SQL, HTML",
            "Databases": "MongoDB, PostgreSQL",
            "Frameworks": "NodeJS, ExpressJS, ReactJS, Kafka, Flask, React-Bootstrap",
            "Cloud & Tools": "Google Cloud Platform (GCP), Docker, Kubernetes, NGINX, Git, Visual Studio, Eclipse, Kafka"
        }
    },
    "experience": {
        "title": "Experience",
        "entries": [
            {
                "role": "Software Development Engineer",
                "company": "Codedale",
                "duration": "April 2025 – Present",
                "location": "Bengaluru, India",
                "highlights": [
                    "Built and maintained 100+ robust backend endpoints using TypeScript and Next.js, reducing API response time by 90% and improving system reliability for 100k users.",
                    "Designed and implemented PostgreSQL schemas and Drizzle ORM models handling 20K+ records, optimizing queries and reducing average database load by 60%.",
                    "Contributed to 50+ production features end-to-end, including API design, authentication, and data validation, supporting 10k+ active users."
                ]
            },
            {
                "role": "Founder & Lead Engineer",
                "company": "Neylon AI",
                "duration": "Aug 2025 – Present",
                "location": "Bengaluru, India",
                "highlights": [
                    "Founded Neylon AI, an AI agency delivering scalable AI assistants and agent-based solutions, serving diverse clients with intelligent automation.",
                    "Developed AI agents using LangGraph, LangChain, and vector databases, implementing a RAG pipeline capable of handling 20K+ data records efficiently.",
                    "Engineered the platform to process and query 10GB+ datasets seamlessly using Django backend and Next.js frontend, ensuring high-performance AI workflows."
                ]
            }
        ]
    },
    "projects": {
        "title": "Projects",
        "entries": [
            {
                "name": "AI-Powered Market Sentiment Analyzer",
                "tech_stack": "Python, React.js, VPS",
                "links": {
                    "Live": "neylonai.vercel.app",
                    "GitHub": "github.com/Hrithik450"
                },
                "highlights": [
                    "Developed an AI-driven sentiment analysis tool that gathers 100,000+ data points daily from crypto news, social media posts, and market trends.",
                    "Integrated Natural Language Processing (NLP) using VADER, achieving 85%+ accuracy in sentiment classification (bullish / bearish).",
                    "Built a scalable backend with Flask, handling 500+ API requests per second while fetching data from RSS feeds, Twitter APIs, and crypto forums.",
                    "Potential Impact: Help traders reduce decision-making time by 30%, improving trade success rates based on sentiment analysis."
                ]
            },
            {
                "name": "MERN Launcher – Automation for MERN Stack Setup",
                "tech_stack": "Bash, Node.js",
                "links": {
                    "Live": "neylonai.vercel.app",
                    "GitHub": "github.com/Hrithik450"
                },
                "highlights": [
                    "Developed an automation tool that streamlines the setup of a fully functional MERN (MongoDB, Express.js, React.js, Node.js) stack with a single command.",
                    "Automated the installation of frontend and backend dependencies, including React Router, Redux, Express, Mongoose, and authentication modules.",
                    "Potential Impact: Reduces setup time from hours to minutes, enabling developers to kickstart MERN projects with minimal effort."
                ]
            }
        ]
    },
    "education": {
        "title": "Education",
        "entries": [
            {
                "institution": "College Of Engineering, Pune",
                "degree": "Bachelor of Technology in Mechanical Engineering",
                "duration": "Aug 2017 – Aug 2020",
                "location": "Bengaluru, India"
            }
        ]
    },
}